# EcoSupport-Copilot — Data Preparation (01)

Outputs:
- data/kb/passages.jsonl
- data/processed/retriever_train.jsonl
- data/processed/reranker_train.jsonl
- data/processed/generator_train.jsonl
- data/processed/dpo_pairs.jsonl
- data/synthetic_tool_labels/tool_train.jsonl

In [ ]:
import os
import re
import json
import random
from typing import Any, Dict, List, Tuple

from datasets import load_dataset
from transformers import AutoTokenizer
from rank_bm25 import BM25Okapi

random.seed(42)

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath(os.getcwd())
DATA_RAW = os.path.join(ROOT, 'data', 'raw')
DATA_KB = os.path.join(ROOT, 'data', 'kb')
DATA_PROC = os.path.join(ROOT, 'data', 'processed')
DATA_SYN = os.path.join(ROOT, 'data', 'synthetic_tool_labels')

for p in [DATA_RAW, DATA_KB, DATA_PROC, DATA_SYN]:
    os.makedirs(p, exist_ok=True)

def write_jsonl(path: str, rows: List[Dict[str, Any]]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def clean_text(text: str) -> str:
    if text is None:
        return ''
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.replace('\u00a0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## 3.1 KB Construction

In [ ]:
# Required download snippet (per spec)
from datasets import load_dataset
bitext = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
msmarco = load_dataset('microsoft/ms_marco', 'v2.1')

# Optional supplement
try:
    faq = load_dataset('Bellerophon/amazon-faq-dataset')
except Exception as e:
    faq = None
    print('Optional FAQ dataset not loaded:', e)

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
MAX_TOKENS = 256
OVERLAP = 32

def chunk_with_offsets(text: str, max_tokens: int = MAX_TOKENS, overlap: int = OVERLAP) -> List[Tuple[str, int, int]]:
    text = clean_text(text)
    if not text:
        return []
    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    offsets = enc.get('offset_mapping', [])
    if not offsets:
        return []
    chunks: List[Tuple[str, int, int]] = []
    start_tok = 0
    step = max(1, max_tokens - overlap)
    while start_tok < len(offsets):
        end_tok = min(len(offsets), start_tok + max_tokens)
        span_start = int(offsets[start_tok][0])
        span_end = int(offsets[end_tok - 1][1])
        chunk_text = text[span_start:span_end].strip()
        if chunk_text:
            chunks.append((chunk_text, span_start, span_end))
        if end_tok >= len(offsets):
            break
        start_tok += step
    return chunks

HF_SOURCE = 'https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset'

def iter_bitext_rows():
    split = bitext['train'] if 'train' in bitext else list(bitext.values())[0]
    for r in split:
        q = r.get('instruction') or r.get('question') or r.get('user') or r.get('query')
        a = r.get('response') or r.get('answer') or r.get('assistant')
        cat = r.get('category') or r.get('intent') or 'general'
        if q and a:
            yield {'question': str(q), 'answer': str(a), 'category': str(cat)}

seen = set()
passages: List[Dict[str, Any]] = []
idx_by_cat: Dict[str, int] = {}

for row in iter_bitext_rows():
    category = re.sub(r'[^a-z0-9]+', '_', row['category'].lower()).strip('_') or 'general'
    ans = clean_text(row['answer'])
    if not ans:
        continue
    key = ans.lower()
    if key in seen:
        continue
    seen.add(key)
    idx_by_cat.setdefault(category, 0)
    base_id = idx_by_cat[category]
    for j, (chunk_text, span_start, span_end) in enumerate(chunk_with_offsets(ans)):
        doc_id = f'KB_{category}_{base_id}_{j}'
        passages.append({'doc_id': doc_id, 'passage_text': chunk_text, 'category': category, 'span_start': span_start, 'span_end': span_end, 'source_url': HF_SOURCE})
    idx_by_cat[category] += 1

passages_path = os.path.join(DATA_KB, 'passages.jsonl')
write_jsonl(passages_path, passages)
print('KB passages:', len(passages), '->', passages_path)

## 3.2 Retriever Training Data
For each Bitext QA pair: query = question; positive = best-matching passage to the answer; hard negatives mined by BM25 over the KB passages.

In [ ]:
passages = read_jsonl(passages_path)
by_doc = {p['doc_id']: p for p in passages}
corpus_ids = [p['doc_id'] for p in passages]
corpus_tokens = [p['passage_text'].lower().split() for p in passages]
bm25 = BM25Okapi(corpus_tokens)

def find_positive_doc_id(answer_text: str) -> str:
    toks = clean_text(answer_text).lower().split()
    if not toks:
        return ''
    scores = bm25.get_scores(toks)
    best_i = int(max(range(len(scores)), key=lambda i: scores[i]))
    return corpus_ids[best_i]

retriever_rows: List[Dict[str, Any]] = []
for row in iter_bitext_rows():
    query = clean_text(row['question'])
    answer = clean_text(row['answer'])
    if not query or not answer:
        continue
    pos_doc_id = find_positive_doc_id(answer)
    if not pos_doc_id:
        continue
    q_toks = query.lower().split()
    scores = bm25.get_scores(q_toks)
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    hard_negs = []
    for i in ranked[:20]:
        did = corpus_ids[i]
        if did == pos_doc_id:
            continue
        hard_negs.append(did)
        if len(hard_negs) >= 5:
            break
    if len(hard_negs) < 5:
        continue
    retriever_rows.append({'query': query, 'positive_doc_id': pos_doc_id, 'negative_doc_ids': hard_negs})

retr_path = os.path.join(DATA_PROC, 'retriever_train.jsonl')
write_jsonl(retr_path, retriever_rows)
print('Retriever rows:', len(retriever_rows), '->', retr_path)

## 3.3 Reranker Training Data
Create (query, passage, label) pairs: label=1 for positive passage, 0 for BM25 hard negatives.

In [ ]:
reranker_rows: List[Dict[str, Any]] = []
for r in retriever_rows:
    q = r['query']
    pos = by_doc[r['positive_doc_id']]['passage_text']
    reranker_rows.append({'query': q, 'passage': pos, 'label': 1})
    for neg_id in r['negative_doc_ids']:
        neg = by_doc[neg_id]['passage_text']
        reranker_rows.append({'query': q, 'passage': neg, 'label': 0})

rer_path = os.path.join(DATA_PROC, 'reranker_train.jsonl')
write_jsonl(rer_path, reranker_rows)
print('Reranker rows:', len(reranker_rows), '->', rer_path)

## 3.4 Generator Training Data
Format instruction-tuning examples with explicit citations.

In [ ]:
def passage_block_for(r: Dict[str, Any]) -> str:
    ids = [r['positive_doc_id']] + r['negative_doc_ids'][:2]
    parts = []
    for did in ids:
        parts.append('[{}] {}'.format(did, by_doc[did]['passage_text']))
    return '\n'.join(parts)

SYSTEM_PERSONA = (
    'You are EcoSupport-Copilot, a helpful e-commerce customer support agent. '
    'Answer concisely, do not hallucinate, and ALWAYS include citations as [doc_id]. '
    'If you are unsure or the question is account/order-specific, escalate by recommending a support ticket.'
)

gen_rows: List[Dict[str, Any]] = []
for r in retriever_rows:
    query = r['query']
    doc_id = r['positive_doc_id']
    passage = by_doc[doc_id]['passage_text']
    passage_block = passage_block_for(r)
    prompt = (
        'Answer the user query using the provided passages. Always cite as [doc_id]. Escalate if unsure.\n\n'
        + 'Passages:\n' + passage_block + '\n\n'
        + 'Query: ' + query + '\n'
    )
    span_text = passage[:200].strip().replace("'", "")
    response = passage + ' [' + doc_id + '] ' + "[Source: {}, span: '{}' ]".format(doc_id, span_text)
    gen_rows.append({
        'prompt': 'System: ' + SYSTEM_PERSONA + '\nUser: ' + prompt + 'Assistant:',
        'response': response,
        'query': query,
        'doc_id': doc_id
    })

gen_path = os.path.join(DATA_PROC, 'generator_train.jsonl')
write_jsonl(gen_path, gen_rows)
print('Generator rows:', len(gen_rows), '->', gen_path)

## 3.5 DPO Preference Data
Generate (chosen, rejected) pairs using a lightweight generator and a rubric score.

In [ ]:
from transformers import pipeline

# Lightweight pre-tuning generator for pair creation (CPU-friendly)
gen_pipe = pipeline('text-generation', model='distilgpt2')

def rubric_score(text: str) -> float:
    t = text or ''
    words = max(1, len(t.split()))
    citation_present = 1.0 if re.search(r'\[[A-Za-z0-9_:-]+\]', t) else 0.0
    conciseness = 1.0 / float(words)
    escalation_correct = 1.0 if re.search(r'(create a ticket|open a ticket|escalat)', t.lower()) else 0.0
    groundedness = 1.0 if citation_present > 0 else 0.0
    return 0.4 * citation_present + 0.2 * conciseness + 0.2 * escalation_correct + 0.2 * groundedness

dpo_rows: List[Dict[str, Any]] = []
for ex in gen_rows[:2000]:
    prompt = ex['prompt']
    out1 = gen_pipe(prompt, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.9)[0]['generated_text']
    out2 = gen_pipe(prompt, max_new_tokens=120, do_sample=True, temperature=1.0, top_p=0.95)[0]['generated_text']
    s1 = rubric_score(out1)
    s2 = rubric_score(out2)
    chosen, rejected = (out1, out2) if s1 >= s2 else (out2, out1)
    dpo_rows.append({'prompt': prompt, 'chosen': chosen, 'rejected': rejected})

dpo_path = os.path.join(DATA_PROC, 'dpo_pairs.jsonl')
write_jsonl(dpo_path, dpo_rows)
print('DPO pairs:', len(dpo_rows), '->', dpo_path)

## 3.6 Tool Policy Data (synthetic)
Heuristic labeling for tool calls and save to data/synthetic_tool_labels/tool_train.jsonl.

In [ ]:
def label_tool(query: str) -> Dict[str, Any]:
    q = (query or '').lower()
    if any(k in q for k in ['policy', 'return', 'refund', 'exchange', 'warranty', 'shipping', 'delivery']):
        if any(k in q for k in ['return', 'refund']):
            section_id = 'RETURN_POLICY'
        elif 'exchange' in q:
            section_id = 'EXCHANGE_POLICY'
        elif 'warranty' in q:
            section_id = 'WARRANTY_POLICY'
        elif 'ship' in q or 'shipping' in q:
            section_id = 'SHIPPING_POLICY'
        elif 'deliver' in q or 'delivery' in q or 'missing' in q or 'lost' in q:
            section_id = 'DELIVERY_ISSUES_POLICY'
        else:
            section_id = 'RETURN_POLICY'
        return {'name': 'GetPolicy', 'args': {'section_id': section_id}, 'reason': 'Policy intent detected.'}
    if any(k in q for k in ['complaint', 'not resolved', 'escalate', 'angry', 'chargeback', 'fraud']):
        return {'name': 'CreateTicket', 'args': {'summary': query[:240], 'category': 'support', 'severity': 'high' if ('fraud' in q or 'chargeback' in q) else 'medium'}, 'reason': 'Escalation intent detected.'}
    if re.search(r'\b(hi|hello|hey|thanks)\b', q) and len(q.split()) <= 5:
        return {'name': 'None', 'args': {}, 'reason': 'Greeting/simple message.'}
    return {'name': 'SearchKB', 'args': {'query': query}, 'reason': 'Default retrieval.'}

tool_rows: List[Dict[str, Any]] = []
for r in retriever_rows[:2000]:
    labeled = label_tool(r['query'])
    tool_rows.append({
        'query': r['query'],
        'passages_context': passage_block_for(r),
        'tool_call': {'name': labeled['name'], 'args': labeled.get('args', {})},
        'reason': labeled.get('reason', '')
    })

tool_path = os.path.join(DATA_SYN, 'tool_train.jsonl')
write_jsonl(tool_path, tool_rows)
print('Tool-policy rows:', len(tool_rows), '->', tool_path)
print('Script version: src/tool_policy/generate_tool_labels.py')